In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import joblib, json, os
from datetime import datetime, timedelta

# PyTorch Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, roc_auc_score

# Create directory for saving models
os.makedirs('model_artifacts', exist_ok=True)

# Set device (Use GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Environment ready. Using device: {device}")

✅ Environment ready. Using device: cpu


In [2]:
def get_advanced_features(ticker="NVDA"):
    df = yf.download(ticker, start="2015-01-01")
    
    # Basic Price Features
    df['Returns'] = df['Close'].pct_change()
    df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
    
    # Technical Indicators (Moving Averages)
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA50'] = df['Close'].rolling(50).mean()
    
    # RSI (Relative Strength Index)
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # Volume Relative
    df['Vol_Rel'] = df['Volume'] / df['Volume'].rolling(20).mean()
    
    # Target: 1 if price is higher in 5 days, else 0
    df['Target'] = (df['Close'].shift(-5) > df['Close']).astype(int)
    
    return df.dropna()

data = get_advanced_features("NVDA")
print(f"✅ Data processed. Total Rows: {len(data)}")

[*********************100%***********************]  1 of 1 completed

✅ Data processed. Total Rows: 2805


In [3]:
SEQ_LEN = 30
feature_cols = ['Close', 'Returns', 'MA20', 'MA50', 'RSI', 'Vol_Rel']

# Scaling
scaler = RobustScaler()
scaled_data = scaler.fit_transform(data[feature_cols])

def create_sequences(data, target, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:(i + seq_len)])
        y.append(target[i + seq_len])
    return np.array(X), np.array(y)

X_seq, y_seq = create_sequences(scaled_data, data['Target'].values, SEQ_LEN)

# Convert to PyTorch Tensors
X_tensor = torch.FloatTensor(X_seq).to(device)
y_tensor = torch.FloatTensor(y_seq).view(-1, 1).to(device)

print(f"✅ Input Shape: {X_tensor.shape}") # (Samples, 30 days, 6 features)

✅ Input Shape: torch.Size([2775, 30, 6])


In [4]:
class StockSageModel(nn.Module):
    def __init__(self, n_features):
        super(StockSageModel, self).__init__()
        
        # Layer 1: CNN (Extracts local patterns)
        self.cnn = nn.Conv1d(in_channels=n_features, out_channels=64, kernel_size=3)
        
        # Layer 2: LSTM (Learns temporal trends)
        self.lstm = nn.LSTM(input_size=64, hidden_size=64, batch_first=True)
        
        # Layer 3 & 4: Dense/Linear Layers
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)
        
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x is (Batch, Seq, Features), CNN needs (Batch, Features, Seq)
        x = x.transpose(1, 2)
        x = self.relu(self.cnn(x))
        x = x.transpose(1, 2)
        
        # LSTM
        out, (hn, cn) = self.lstm(x)
        
        # Take the last time step output
        x = self.relu(self.fc1(out[:, -1, :]))
        x = self.sigmoid(self.fc2(x))
        return x

model = StockSageModel(len(feature_cols)).to(device)
print(model)

StockSageModel(
  (cnn): Conv1d(6, 64, kernel_size=(3,), stride=(1,))
  (lstm): LSTM(64, 64, batch_first=True)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)


In [5]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Simple DataLoader
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

print("🚀 Training Started...")
for epoch in range(20):
    model.train()
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/20], Loss: {loss.item():.4f}")

# Save the model
torch.save(model.state_dict(), 'model_artifacts/best_model.pth')
joblib.dump(scaler, 'model_artifacts/scaler.pkl')

# Save Metadata for app.py
metadata = {
    "feature_cols": feature_cols,
    "seq_len": SEQ_LEN,
    "framework": "pytorch"
}
with open('model_artifacts/metadata.json', 'w') as f:
    json.dump(metadata, f)

print("✅ Model saved as 'best_model.pth'")

🚀 Training Started...
Epoch [5/20], Loss: 0.6619
Epoch [10/20], Loss: 0.6402
Epoch [15/20], Loss: 0.6098
Epoch [20/20], Loss: 0.4156
✅ Model saved as 'best_model.pth'


In [6]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Simple DataLoader
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

print("🚀 Training Started...")
for epoch in range(20):
    model.train()
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/20], Loss: {loss.item():.4f}")

# Save the model
torch.save(model.state_dict(), 'model_artifacts/best_model.pth')
joblib.dump(scaler, 'model_artifacts/scaler.pkl')

# Save Metadata for app.py
metadata = {
    "feature_cols": feature_cols,
    "seq_len": SEQ_LEN,
    "framework": "pytorch"
}
with open('model_artifacts/metadata.json', 'w') as f:
    json.dump(metadata, f)

print("✅ Model saved as 'best_model.pth'")

🚀 Training Started...
Epoch [5/20], Loss: 0.2952
Epoch [10/20], Loss: 0.2869
Epoch [15/20], Loss: 0.4438
Epoch [20/20], Loss: 0.2384
✅ Model saved as 'best_model.pth'
